## Gold Cascade Modeling (Deliverable 1.3.3)

This notebook implements the full three-stage cascade anomaly detection pipeline. The objective is to determine whether a layered approach improves alert quality when compared to the single-model baseline.

**Purpose:**  
To operationalize the cascade architecture defined in Section C, generating alert outputs for each stage of the cascade: broad detection (Stage 1), refined detection (Stage 2), and rule/profile/historical confirmation (Stage 3).

**Stages Implemented:**

1. **Stage 1 — Broad Isolation Forest**  
   High-sensitivity anomaly screen using all 50 Gold numeric features.

2. **Stage 2 — Narrow Isolation Forest**  
   Secondary detector trained on the reduced feature subset identified during Silver EDA.

3. **Stage 3 — Rule / Profile / Historical Confirmation**  
   Final confirmation based on behavior profiles, persistence checks, drift features, and cross-sensor consistency.

**Key Goals:**

- Load the Gold preprocessed dataset and Gold feature artifacts.
- Train Stage 1 and Stage 2 Isolation Forest models.
- Apply Stage 3 rule/profile confirmation logic based on Silver EDA outputs.
- Generate and store alert outputs for all three cascade stages.
- Export all cascade artifacts for comparative evaluation.

**Relevance to Section C:**  
This notebook produces the layered alert outputs required for evaluating the cascade’s effect on false positives, noisy alerts, and anomaly sensitivity. These outputs are necessary for the statistical tests, alert-volume comparisons, and visual communication described in Section C.

In [37]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

#import os
#import glob

from pathlib import Path
import yaml
import re

import logging
import wandb

import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

import joblib 

from sklearn.model_selection import train_test_split, KFold

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, RobustScaler

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score

from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

import pyarrow.parquet as pq
import pyarrow as pa


import hashlib


# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json, load_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging
from utils.wandb_utils import finalize_wandb_stage

# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


In [38]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [39]:
def log_gold_paths(paths, logger: logging.Logger) -> None:
    logger.info("Project Root Path Loaded: %s", paths.root)
    logger.info("Project Logging Path Loaded: %s", paths.logs)
    logger.info("Project Artifacts Path Loaded: %s", paths.artifacts)
    logger.info("Project Notebooks Path Loaded %s", paths.notebooks)
    logger.info("Project Data Path Loaded: %s", paths.data)
    logger.info("Data Bronze Path Loaded: %s", paths.data_bronze)
    logger.info("Data Bronze Training Path Loaded: %s", paths.data_bronze_train)
    logger.info("Data Bronze Testing Path Loaded: %s", paths.data_bronze_test)
    logger.info("Data Silver Path Loaded: %s", paths.data_silver)
    logger.info("Data Silver Training Path Loaded: %s", paths.data_silver_train)
    logger.info("Data Silver Testing Path Loaded: %s", paths.data_silver_test)
    logger.info("Data Gold Path Loaded: %s", paths.data_gold)

In [40]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Configurables

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Stage Details
STAGE = "gold"
LAYER_NAME = "gold"
GOLD_VERSION = "gold__001"
RECIPE_ID = "gold_modeling__v001_cascade"


DATASET_NAME_CONFIG = "pump"
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Weights and Biases
WANDB_PROJECT = "capstone"
WANDB_ENTITY = "dcoo230-western-governors-university"
WANDB_RUN_NAME = f"{GOLD_VERSION}"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# File names
GOLD_FIT_FILE_NAME = f"{DATASET_NAME}__gold__fit_normal_only.parquet"
GOLD_TRAIN_FILE_NAME = f"{DATASET_NAME}__gold__train.parquet"
GOLD_TEST_FILE_NAME = f"{DATASET_NAME}__gold__test.parquet"

STAGE1_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage1_features.json"
STAGE2_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage2_features.json"
STAGE3_PRIMARY_FILE_NAME = f"{DATASET_NAME}__gold__stage3_primary_rule_sensors.json"
STAGE3_SECONDARY_FILE_NAME = f"{DATASET_NAME}__gold__stage3_secondary_rule_sensors.json"


STAGE1_MODEL_FILE_NAME = f"{DATASET_NAME}__gold__stage1_isolation_forest.joblib"
STAGE2_MODEL_FILE_NAME = f"{DATASET_NAME}__gold__stage2_isolation_forest.joblib"



CASCADE_THRESHOLDS_FILE_NAME = f"{DATASET_NAME}__gold__cascade_thresholds.json"
CASCADE_SUMMARY_FILE_NAME = f"{DATASET_NAME}__gold__cascade_summary.json"
CASCADE_METADATA_FILE_NAME = f"{DATASET_NAME}__gold__cascade_metadata.json"


CASCADE_RESULTS_FILE_NAME = f"{DATASET_NAME}__gold__cascade_results.csv"
COMPARISON_FILE_NAME = f"{DATASET_NAME}__gold__comparison__baseline_vs_cascade.csv"

CASCADE_REFERENCE_PROFILE_FILE_NAME = f"{DATASET_NAME}__gold__cascade_reference_profile.csv"



#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

RANDOM_SEED = 42

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Cascade thresholding
STAGE1_THRESHOLD_PERCENTILE = 95.0   # broader / more sensitive
STAGE2_THRESHOLD_PERCENTILE = 98.5   # narrower / stricter

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Isolation Forest size
STAGE1_ESTIMATOR_COUNT = 200
STAGE2_ESTIMATOR_COUNT = 300

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


GOLD_CASCADE_LEDGER_FILE_NAME = f"ledger__{DATASET_NAME}__gold_cascade_modeling.json"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


In [42]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Paths Setup

# Get Path's Object
paths = get_paths()

# Gold
GOLD_FIT_DATA_PATH = paths.data_gold / GOLD_FIT_FILE_NAME
GOLD_TRAIN_DATA_PATH = paths.data_gold / GOLD_TRAIN_FILE_NAME
GOLD_TEST_DATA_PATH = paths.data_gold / GOLD_TEST_FILE_NAME
GOLD_ARTIFACTS_PATH = paths.artifacts / "gold" / DATASET_NAME

MODELS_PATH = paths.models / DATASET_NAME

# Stage artifacts
STAGE1_FEATURES_PATH = GOLD_ARTIFACTS_PATH / STAGE1_FEATURES_FILE_NAME
STAGE2_FEATURES_PATH = GOLD_ARTIFACTS_PATH / STAGE2_FEATURES_FILE_NAME
STAGE3_PRIMARY_PATH = GOLD_ARTIFACTS_PATH / STAGE3_PRIMARY_FILE_NAME
STAGE3_SECONDARY_PATH = GOLD_ARTIFACTS_PATH / STAGE3_SECONDARY_FILE_NAME

# Outputs
CASCADE_RESULTS_PATH = GOLD_ARTIFACTS_PATH / CASCADE_RESULTS_FILE_NAME

STAGE1_MODEL_ARTIFACT_PATH = GOLD_ARTIFACTS_PATH / STAGE1_MODEL_FILE_NAME 
STAGE2_MODEL_ARTIFACT_PATH = GOLD_ARTIFACTS_PATH / STAGE2_MODEL_FILE_NAME 

STAGE1_MODELS_PATH = MODELS_PATH / STAGE1_MODEL_FILE_NAME
STAGE2_MODELS_PATH = GOLD_ARTIFACTS_PATH / STAGE2_MODEL_FILE_NAME


CASCADE_THRESHOLDS_PATH = GOLD_ARTIFACTS_PATH / CASCADE_THRESHOLDS_FILE_NAME
CASCADE_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / CASCADE_SUMMARY_FILE_NAME
CASCADE_METADATA_PATH = GOLD_ARTIFACTS_PATH / CASCADE_METADATA_FILE_NAME

CASCADE_REFERENCE_PROFILE_PATH = GOLD_ARTIFACTS_PATH / CASCADE_REFERENCE_PROFILE_FILE_NAME 

# Logs
LOGS_PATH = paths.logs

# Path Failsafes
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)



In [44]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [45]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_modeling_cascade.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_gold_paths(paths, logger)


2026-03-02 05:21:09,542 | INFO | capstone.gold | Gold Modeling stage starting
2026-03-02 05:21:09,546 | INFO | capstone.gold | Project Root Path Loaded: /workspace
2026-03-02 05:21:09,548 | INFO | capstone.gold | Project Logging Path Loaded: /workspace/logs
2026-03-02 05:21:09,551 | INFO | capstone.gold | Project Artifacts Path Loaded: /workspace/artifacts
2026-03-02 05:21:09,553 | INFO | capstone.gold | Project Notebooks Path Loaded /workspace/notebooks
2026-03-02 05:21:09,557 | INFO | capstone.gold | Project Data Path Loaded: /workspace/data
2026-03-02 05:21:09,559 | INFO | capstone.gold | Data Bronze Path Loaded: /workspace/data/bronze
2026-03-02 05:21:09,561 | INFO | capstone.gold | Data Bronze Training Path Loaded: /workspace/data/bronze/train
2026-03-02 05:21:09,563 | INFO | capstone.gold | Data Bronze Testing Path Loaded: /workspace/data/bronze/test
2026-03-02 05:21:09,565 | INFO | capstone.gold | Data Silver Path Loaded: /workspace/data/silver
2026-03-02 05:21:09,568 | INFO | c

In [46]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [47]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_modeling_cascade",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "stage1_threshold_percentile": STAGE1_THRESHOLD_PERCENTILE,
        "stage2_threshold_percentile": STAGE2_THRESHOLD_PERCENTILE,
        "gold_input_path": str(GOLD_TRAIN_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


2026-03-02 05:21:18,268 | INFO | capstone.gold | W&B initialized: gold__001


In [48]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [49]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


2026-03-02 05:21:18,298 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T05:21:18.298831+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger', 'why': None, 'consequence': None, 'data': {}}


{'ts_utc': '2026-03-02T05:21:18.298831+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'init',
 'message': 'Initialized ledger',
 'why': None,
 'consequence': None,
 'data': {}}

In [50]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [51]:
logger.info("Loading Gold fit parquet: %s", GOLD_FIT_DATA_PATH)
gold_fit_dataframe = load_data(GOLD_FIT_DATA_PATH)

logger.info("Loading Gold test parquet: %s", GOLD_TEST_DATA_PATH)
gold_test_dataframe = load_data(GOLD_TEST_DATA_PATH)

logger.info("Loading Stage 1 features: %s", STAGE1_FEATURES_PATH)
stage1_feature_columns = load_json(STAGE1_FEATURES_PATH)

logger.info("Loading Stage 2 features: %s", STAGE2_FEATURES_PATH)
stage2_feature_columns = load_json(STAGE2_FEATURES_PATH)

logger.info("Loading Stage 3 primary rule sensors: %s", STAGE3_PRIMARY_PATH)
stage3_primary_rule_sensors = load_json(STAGE3_PRIMARY_PATH)

logger.info("Loading Stage 3 secondary rule sensors: %s", STAGE3_SECONDARY_PATH)
stage3_secondary_rule_sensors = load_json(STAGE3_SECONDARY_PATH)

ledger.add(
    kind="step",
    step="load_modeling_inputs",
    message="Loaded saved Gold fit/test parquet artifacts and Stage 1/2/3 feature inputs.",
    data={
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_test_path": str(GOLD_TEST_DATA_PATH),
        "gold_fit_shape": list(gold_fit_dataframe.shape),
        "gold_test_shape": list(gold_test_dataframe.shape),
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_count": int(len(stage3_secondary_rule_sensors)),
    },
    logger=logger,
)

gold_test_dataframe.head(3)

2026-03-02 05:21:18,341 | INFO | capstone.gold | Loading Gold fit parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-03-02 05:21:18,347 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-03-02 05:21:19,044 | INFO | capstone.gold | Loading Gold test parquet: /workspace/data/gold/pump__gold__test.parquet
2026-03-02 05:21:19,047 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__test.parquet
2026-03-02 05:21:19,200 | INFO | capstone.gold | Loading Stage 1 features: /workspace/artifacts/gold/pump/pump__gold__stage1_features.json
2026-03-02 05:21:19,207 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/pump__gold__stage1_features.json
2026-03-02 05:21:19,216 | INFO | capstone.gold | Loading Stage 2 features: /workspace/artifacts/gold/pump/pump__gold__stage2_features.json
2026-03-02 05:21:19,224 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/

,meta__asset_id,meta__cleaning_recipe_id,meta__dataset,meta__dataset_name,meta__dataset_source,meta__event_id,meta__event_time_parse_success_percent,meta__event_time_source,meta__feature_count,meta__feature_set_id,meta__has_label_candidates,meta__has_status_candidates,meta__ingested_at_utc,meta__label_source,meta__label_source_column,meta__label_source_kind,meta__label_type,meta__layer,meta__processed_at_utc,meta__record_id,meta__run_id,meta__silver_version,meta__source_file,meta__source_row_id,meta__split,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status,meta__gold_imputation,meta__gold_scaler
0,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:176256,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,gold,2026-03-01 04:18:23.781532+00:00,5992450321897436818,run__001,silver__001,sensor.csv,176256,unsplit,2018-08-01 09:36:00+00:00,176256,176256,2018-08-01 00:00:00+00:00,0,0,1,NORMAL,2.453588,49.348957,51.51910,44.53125,642.129639,70.64276,14.22888,16.65220,15.14757,15.38628,43.55097,52.65041,34.85802,19.25599,420.7400,465.5606,473.3555,2.672640,668.1029,400.0775,881.1509,532.5089,1092.967,629.4528,744.5596,977.3638,485.1414,941.6569,596.5172,685.6481,961.4583,1011.959,557.5650,319.7289,476.5692,813.5801,28.63851,46.09375,34.37500,75.52083,32.55208,34.11458,44.27083,47.45370,42.24537,49.768520,42.53472,312.5000,66.84028,210.9375,2018-08-01 09:36:00,NORMAL,forward_fill_within_group_then_median,none
1,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:176257,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,gold,2026-03-01 04:18:23.781532+00:00,2078900024144198128,run__001,silver__001,sensor.csv,176257,unsplit,2018-08-01 09:37:00+00:00,176257,176257,2018-08-01 00:00:00+00:00,0,0,1,NORMAL,2.453588,49.348960,51.60590,44.53125,630.786987,71.26656,14.39525,16.75347,15.18374,15.16204,45.64900,52.03297,34.67496,19.94810,420.0238,462.5600,465.0322,2.573149,665.9155,399.6319,879.0721,534.0840,1093.140,626.4788,739.2676,980.0061,496.7717,941.4617,539.0922,716.2037,952.6041,1009.355,558.2068,325.1864,492.9296,813.3455,30.54268,45.83333,34.37500,83.07291,32.55208,33.85416,44.01041,47.74306,41.95602,49.768517,43.11343,318.2870,66.55093,209.4907,2018-08-01 09:37:00,NORMAL,forward_fill_within_group_then_median,none
2,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:176258,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,gold,2026-03-01 04:18:23.781532+00:00,5178773400662276631,run__001,silver__001,sensor.csv,176258,unsplit,2018-08-01 09:38:00+00:00,176258,176258,2018-08-01 00:00:00+00:00,0,0,1,NORMAL,2.451620,49.305550,51.47569,44.57465,634.606445,70.15800,14.43866,16.78964,15.18374,15.11140,45.16984,51.57830,34.54854,19.92392,421.2197,463.6213,462.7087,2.522767,666.9105,398.2779,882.9197,534.3575,1092.486,630.1271,742.2440,979.2006,507.1257,953.0093,552.8891,701.3889,995.3124,1021.922,567.2665,317.8607,486.9266,804.4023,41.09998,45.83333,35.41666,90.36458,32.29166,33.85416,45.57291,47.45370,41.66667,49.768520,43.11343,320.0231,66.26157,206.5972,2018-08-01 09:38:00,NORMAL,forward_fill_withi

In [52]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [53]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> pd.DataFrame:
    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]

        reference_rows.append({
            "feature_name": feature_name,
            "median_value": float(feature_series.median()),
            "mean_value": float(feature_series.mean()),
            "standard_deviation": float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0,
            "lower_bound": float(feature_series.quantile(0.05)),
            "upper_bound": float(feature_series.quantile(0.95)),
        })

    reference_profile = pd.DataFrame(reference_rows)
    return reference_profile

In [54]:
reference_profile_features = list(dict.fromkeys(
    stage1_feature_columns + stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

reference_profile = build_reference_profile(
    gold_fit_dataframe,
    feature_columns=reference_profile_features,
)

ledger.add(
    kind="step",
    step="build_reference_profile",
    message="Built fit-period reference profile for Stage 3 confirmation logic.",
    data={
        "training_rows": int(len(gold_fit_dataframe)),
        "reference_feature_count": int(len(reference_profile_features)),
    },
    logger=logger,
)

reference_profile.head(10)

2026-03-02 05:21:19,786 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T05:21:19.786267+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'build_reference_profile', 'message': 'Built fit-period reference profile for Stage 3 confirmation logic.', 'why': None, 'consequence': None, 'data': {'training_rows': 161772, 'reference_feature_count': 50}}


,feature_name,median_value,mean_value,standard_deviation,lower_bound,upper_bound
0,sensor_00,2.456539,2.410299,0.271235,2.203704,2.512616
1,sensor_01,47.960068,47.912636,2.318929,44.227430,51.866320
2,sensor_02,51.736110,51.613350,2.028793,48.654510,54.296871
3,sensor_03,44.053818,43.934741,1.727763,41.102428,46.614580
4,sensor_04,632.754600,623.225797,56.928953,595.370400,644.444500
5,sensor_05,76.974545,76.514339,9.794771,63.915637,90.507114
6,sensor_06,13.592300,13.599493,0.769163,13.028070,14.605030
7,sensor_07,16.160300,16.076092,0.625054,15.588830,16.702840
8,sensor_08,15.371820,15.411662,0.625472,14.879920,16.160300
9,sensor_09,15.082470,15.072625,0.620664,14.793110,15.632230


In [55]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [56]:
stage1_train_fit_features = gold_fit_dataframe[stage1_feature_columns].values
stage1_test_features = gold_test_dataframe[stage1_feature_columns].values

stage2_train_fit_features = gold_fit_dataframe[stage2_feature_columns].values
stage2_test_features = gold_test_dataframe[stage2_feature_columns].values

if "anomaly_flag" in gold_test_dataframe.columns:
    test_labels = gold_test_dataframe["anomaly_flag"].astype(int).values
else:
    test_labels = None

In [57]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [58]:
def compute_anomaly_scores_isolation_forest(
    model: IsolationForest,
    feature_matrix: np.ndarray,
) -> np.ndarray:
    scores = model.score_samples(feature_matrix)
    anomaly_scores = -scores
    return anomaly_scores

In [59]:
def choose_threshold_by_percentile(
    anomaly_scores: np.ndarray,
    percentile: float,
) -> float:
    return float(np.percentile(anomaly_scores, percentile))

In [60]:

def evaluate_against_labels(
    true_labels: np.ndarray,
    anomaly_scores: np.ndarray,
    threshold: float,
) -> dict:
    predicted_labels = (anomaly_scores >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        average="binary",
        zero_division=0,
    )

    results = {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

    if len(np.unique(true_labels)) == 2:
        results["roc_auc"] = float(roc_auc_score(true_labels, anomaly_scores))
        results["pr_auc"] = float(average_precision_score(true_labels, anomaly_scores))
    else:
        results["roc_auc"] = None
        results["pr_auc"] = None

    return results

In [61]:
stage1_model = IsolationForest(
    n_estimators=STAGE1_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

stage1_model.fit(stage1_train_fit_features)

stage1_train_scores = compute_anomaly_scores_isolation_forest(
    stage1_model,
    stage1_train_fit_features,
)

stage1_test_scores = compute_anomaly_scores_isolation_forest(
    stage1_model,
    stage1_test_features,
)

stage1_threshold = choose_threshold_by_percentile(
    stage1_train_scores,
    STAGE1_THRESHOLD_PERCENTILE,
)

stage1_flags = (stage1_test_scores >= stage1_threshold).astype(int)

stage1_summary = {
    "threshold_percentile": STAGE1_THRESHOLD_PERCENTILE,
    "threshold": float(stage1_threshold),
    "alert_count_all_rows": int(stage1_flags.sum()),
    "alert_count_test_rows": int(stage1_flags.sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage1",
    message="Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored the saved Gold test data.",
    data={
        "estimator_count": int(STAGE1_ESTIMATOR_COUNT),
        "threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "threshold": float(stage1_threshold),
        "feature_count": int(len(stage1_feature_columns)),
        "alert_count_test_rows": int(stage1_summary["alert_count_test_rows"]),
    },
    logger=logger,
)

stage1_summary

2026-03-02 05:21:21,898 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T05:21:21.898511+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage1', 'message': 'Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored the saved Gold test data.', 'why': None, 'consequence': None, 'data': {'estimator_count': 200, 'threshold_percentile': 95.0, 'threshold': 0.532497962842102, 'feature_count': 50, 'alert_count_test_rows': 785}}


{'threshold_percentile': 95.0,
 'threshold': 0.532497962842102,
 'alert_count_all_rows': 785,
 'alert_count_test_rows': 785}

In [62]:
stage2_model = IsolationForest(
    n_estimators=STAGE2_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

stage2_model.fit(stage2_train_fit_features)

stage2_train_scores = compute_anomaly_scores_isolation_forest(
    stage2_model,
    stage2_train_fit_features,
)

stage2_test_scores = compute_anomaly_scores_isolation_forest(
    stage2_model,
    stage2_test_features,
)

stage2_threshold = choose_threshold_by_percentile(
    stage2_train_scores,
    STAGE2_THRESHOLD_PERCENTILE,
)

stage2_raw_flags = (stage2_test_scores >= stage2_threshold).astype(int)
stage2_flags = ((stage1_flags == 1) & (stage2_raw_flags == 1)).astype(int)

stage2_summary = {
    "threshold_percentile": STAGE2_THRESHOLD_PERCENTILE,
    "threshold": float(stage2_threshold),
    "raw_alert_count_all_rows": int(stage2_raw_flags.sum()),
    "stage2_confirmed_count_all_rows": int(stage2_flags.sum()),
    "stage2_confirmed_count_test_rows": int(stage2_flags.sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage2",
    message="Ran Stage 2 narrow Isolation Forest on saved Gold fit data and gated test results through Stage 1.",
    data={
        "estimator_count": int(STAGE2_ESTIMATOR_COUNT),
        "threshold_percentile": float(STAGE2_THRESHOLD_PERCENTILE),
        "threshold": float(stage2_threshold),
        "feature_count": int(len(stage2_feature_columns)),
        "raw_alert_count_all_rows": int(stage2_summary["raw_alert_count_all_rows"]),
        "stage2_confirmed_count_test_rows": int(stage2_summary["stage2_confirmed_count_test_rows"]),
    },
    logger=logger,
)

stage2_summary

2026-03-02 05:21:24,664 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T05:21:24.664687+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage2', 'message': 'Ran Stage 2 narrow Isolation Forest on saved Gold fit data and gated test results through Stage 1.', 'why': None, 'consequence': None, 'data': {'estimator_count': 300, 'threshold_percentile': 98.5, 'threshold': 0.6900805433271796, 'feature_count': 12, 'raw_alert_count_all_rows': 2, 'stage2_confirmed_count_test_rows': 2}}


{'threshold_percentile': 98.5,
 'threshold': 0.6900805433271796,
 'raw_alert_count_all_rows': 2,
 'stage2_confirmed_count_all_rows': 2,
 'stage2_confirmed_count_test_rows': 2}

In [63]:
cascade_results = gold_test_dataframe.copy()

cascade_results["stage1_score"] = stage1_test_scores
cascade_results["stage1_flag"] = stage1_flags

cascade_results["stage2_score"] = stage2_test_scores
cascade_results["stage2_raw_flag"] = stage2_raw_flags
cascade_results["stage2_flag"] = stage2_flags

In [64]:
def compute_profile_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name").to_dict("index")
    breach_counts: list[int] = []

    for _, row_values in dataframe.iterrows():
        row_breach_count = 0

        for feature_name in feature_columns:
            if feature_name not in reference_lookup:
                continue

            current_value = row_values[feature_name]
            if pd.isna(current_value):
                continue

            lower_bound = reference_lookup[feature_name]["lower_bound"]
            upper_bound = reference_lookup[feature_name]["upper_bound"]

            if current_value < lower_bound or current_value > upper_bound:
                row_breach_count += 1

        breach_counts.append(row_breach_count)

    return pd.Series(breach_counts, index=dataframe.index, name="stage3_profile_breach_count")

In [65]:
cascade_results["stage3_profile_breach_count"] = compute_profile_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_primary_rule_sensors,
)

In [66]:
def compute_secondary_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name").to_dict("index")
    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in reference_lookup:
            continue

        lower_bound = reference_lookup[feature_name]["lower_bound"]
        upper_bound = reference_lookup[feature_name]["upper_bound"]

        feature_breach_flag = (
            (dataframe[feature_name] < lower_bound) |
            (dataframe[feature_name] > upper_bound)
        ).astype(int)

        breach_counts = breach_counts + feature_breach_flag

    breach_counts.name = "stage3_secondary_breach_count"
    return breach_counts

In [67]:
cascade_results["stage3_secondary_breach_count"] = compute_secondary_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_secondary_rule_sensors,
)

In [68]:
def compute_persistence_flag(
    source_flags: pd.Series,
    *,
    rolling_window_size: int = 3,
    minimum_flags_in_window: int = 2,
) -> pd.Series:
    persistence_flag = (
        source_flags
        .rolling(window=rolling_window_size, min_periods=1)
        .sum()
        .ge(minimum_flags_in_window)
        .astype(int)
    )

    persistence_flag.name = "stage3_persistence_flag"
    return persistence_flag

In [69]:
cascade_results["stage3_persistence_flag"] = compute_persistence_flag(
    cascade_results["stage2_flag"],
    rolling_window_size=3,
    minimum_flags_in_window=2,
)


In [70]:
def compute_drift_flag(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
    rolling_window_size: int = 5,
    drift_threshold_multiplier: float = 1.0,
) -> pd.Series:
    drift_trigger_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]
        feature_standard_deviation = feature_series.std()

        if pd.isna(feature_standard_deviation) or feature_standard_deviation == 0:
            continue

        rolling_median = feature_series.rolling(window=rolling_window_size, min_periods=1).median()
        rolling_delta = (feature_series - rolling_median).abs()

        feature_drift_flag = (
            rolling_delta > (feature_standard_deviation * drift_threshold_multiplier)
        ).astype(int)

        drift_trigger_counts = drift_trigger_counts + feature_drift_flag

    drift_flag = (drift_trigger_counts >= 1).astype(int)
    drift_flag.name = "stage3_drift_flag"
    return drift_flag

In [71]:
stage3_rule_watch_features = list(dict.fromkeys(
    stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

cascade_results["stage3_drift_flag"] = compute_drift_flag(
    cascade_results,
    feature_columns=stage3_rule_watch_features,
    rolling_window_size=5,
    drift_threshold_multiplier=1.0,
)

In [72]:
cascade_results["stage3_profile_breach_flag"] = (
    cascade_results["stage3_profile_breach_count"] >= 2
).astype(int)

cascade_results["stage3_corroboration_flag"] = (
    cascade_results["stage3_secondary_breach_count"] >= 1
).astype(int)

cascade_results["stage3_rule_evidence_count"] = (
    cascade_results["stage3_profile_breach_flag"] +
    cascade_results["stage3_persistence_flag"] +
    cascade_results["stage3_drift_flag"] +
    cascade_results["stage3_corroboration_flag"]
)

cascade_results["cascade_final_flag"] = (
    (cascade_results["stage1_flag"] == 1) &
    (cascade_results["stage2_flag"] == 1) &
    (
        (cascade_results["stage3_profile_breach_count"] >= 2) |
        (cascade_results["stage3_rule_evidence_count"] >= 2)
    )
).astype(int)

stage3_summary = {
    "primary_rule_sensor_count": int(len(stage3_primary_rule_sensors)),
    "secondary_rule_sensor_count": int(len(stage3_secondary_rule_sensors)),
    "profile_breach_rows": int((cascade_results["stage3_profile_breach_flag"] == 1).sum()),
    "corroboration_rows": int((cascade_results["stage3_corroboration_flag"] == 1).sum()),
    "persistence_rows": int((cascade_results["stage3_persistence_flag"] == 1).sum()),
    "drift_rows": int((cascade_results["stage3_drift_flag"] == 1).sum()),
    "final_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "final_alert_count_test_rows": int(cascade_results["cascade_final_flag"].sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage3_confirmation",
    message="Applied Stage 3 confirmation checks to the saved Gold test data.",
    data=stage3_summary,
    logger=logger,
)

cascade_results.head(5)

2026-03-02 05:21:30,827 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T05:21:30.827263+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage3_confirmation', 'message': 'Applied Stage 3 confirmation checks to the saved Gold test data.', 'why': None, 'consequence': None, 'data': {'primary_rule_sensor_count': 5, 'secondary_rule_sensor_count': 7, 'profile_breach_rows': 6694, 'corroboration_rows': 15609, 'persistence_rows': 2, 'drift_rows': 6978, 'final_alert_count_all_rows': 2, 'final_alert_count_test_rows': 2}}


,meta__asset_id,meta__cleaning_recipe_id,meta__dataset,meta__dataset_name,meta__dataset_source,meta__event_id,meta__event_time_parse_success_percent,meta__event_time_source,meta__feature_count,meta__feature_set_id,meta__has_label_candidates,meta__has_status_candidates,meta__ingested_at_utc,meta__label_source,meta__label_source_column,meta__label_source_kind,meta__label_type,meta__layer,meta__processed_at_utc,meta__record_id,meta__run_id,meta__silver_version,meta__source_file,meta__source_row_id,meta__split,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status,meta__gold_imputation,meta__gold_scaler,stage1_score,stage1_flag,stage2_score,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_persistence_flag,stage3_drift_flag,stage3_profile_breach_flag,stage3_corroboration_flag,stage3_rule_evidence_count,cascade_final_flag
0,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:176256,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,gold,2026-03-01 04:18:23.781532+00:00,5992450321897436818,run__001,silver__001,sensor.csv,176256,unsplit,2018-08-01 09:36:00+00:00,176256,176256,2018-08-01 00:00:00+00:00,0,0,1,NORMAL,2.453588,49.348957,51.51910,44.53125,642.129639,70.64276,14.22888,16.65220,15.14757,15.38628,43.55097,52.65041,34.85802,19.25599,420.7400,465.5606,473.3555,2.672640,668.1029,400.0775,881.1509,532.5089,1092.967,629.4528,744.5596,977.3638,485.1414,941.6569,596.5172,685.6481,961.4583,1011.9590,557.5650,319.7289,476.5692,813.5801,28.63851,46.09375,34.37500,75.52083,32.552080,34.114580,44.27083,47.45370,42.24537,49.768520,42.534720,312.5000,66.84028,210.9375,2018-08-01 09:36:00,NORMAL,forward_fill_within_group_then_median,none,0.390684,0,0.393230,0,0,0,0,0,0,0,0,0,0
1,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:176257,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,gold,2026-03-01 04:18:23.781532+00:00,2078900024144198128,run__001,silver__001,sensor.csv,176257,unsplit,2018-08-01 09:37:00+00:00,176257,176257,2018-08-01 00:00:00+00:00,0,0,1,NORMAL,2.453588,49.348960,51.60590,44.53125,630.786987,71.26656,14.39525,16.75347,15.18374,15.16204,45.64900,52.03297,34.67496,19.94810,420.0238,462.5600,465.0322,2.573149,665.9155,399.6319,879.0721,534.0840,1093.140,626.4788,739.2676,980.0061,496.7717,941.4617,539.0922,716.2037,952.6041,1009.3550,558.2068,325.1864,492.9296,813.3455,30.54268,45.83333,34.37500,83.07291,32.552080,33.854160,44.01041,47.74306,41.95602,49.768517,43.113430,318.2870,66.55093,209.4907,2018-08-01 09:37:00,NORMAL,forward_fill_within_group_then_median,none,0.391546,0,0.387259,0,0,1,0,0,0,0,0,0,0
2,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:176258,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,gold,2026-03-01 04:18:23.781532+00:00,5178773400662276631,run__001,silver__001,sensor.csv,176258,unsplit,2018-08-01 09:38:00+00:00,176258,176258,2018-08-01 00:00:00+00:00,0,0,1,NORMAL,2.451620,49.305550,51.47569,44.57465,634.606445,70.15800,14.43866,16.78964,15.18374,15.11140,45.16984,51.57830,34.54854,19.92392,421.2197,463.6213

In [73]:
cascade_metrics = {
    "model": "3-Stage Cascade",
    "stage1_alert_count_all_rows": int(cascade_results["stage1_flag"].sum()),
    "stage2_alert_count_all_rows": int(cascade_results["stage2_flag"].sum()),
    "final_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "stage1_alert_count_test_rows": int(cascade_results["stage1_flag"].sum()),
    "stage2_alert_count_test_rows": int(cascade_results["stage2_flag"].sum()),
    "final_alert_count_test_rows": int(cascade_results["cascade_final_flag"].sum()),
}

if test_labels is not None:
    cascade_test_flags = cascade_results["cascade_final_flag"].astype(int).values

    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels,
        cascade_test_flags,
        average="binary",
        zero_division=0,
    )

    cascade_metrics["precision"] = float(precision)
    cascade_metrics["recall"] = float(recall)
    cascade_metrics["f1"] = float(f1)

cascade_metrics

{'model': '3-Stage Cascade',
 'stage1_alert_count_all_rows': 785,
 'stage2_alert_count_all_rows': 2,
 'final_alert_count_all_rows': 2,
 'stage1_alert_count_test_rows': 785,
 'stage2_alert_count_test_rows': 2,
 'final_alert_count_test_rows': 2,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0}

In [ ]:
stage1_alert_count_all_rows = int(cascade_results["stage1_flag"].sum())
stage2_alert_count_all_rows = int(cascade_results["stage2_flag"].sum())
final_cascade_alert_count_all_rows = int(cascade_results["cascade_final_flag"].sum())

stage1_alert_count_test_rows = int(cascade_results["stage1_flag"].sum())
stage2_alert_count_test_rows = int(cascade_results["stage2_flag"].sum())
final_cascade_alert_count_test_rows = int(cascade_results["cascade_final_flag"].sum())

cascade_thresholds = {
    "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
    "stage1_threshold": float(stage1_threshold),
    "stage2_threshold_percentile": float(STAGE2_THRESHOLD_PERCENTILE),
    "stage2_threshold": float(stage2_threshold),
}

cascade_summary = {
    "dataset_name": DATASET_NAME,
    "cascade_metrics": cascade_metrics,
    "stage1_alert_count_all_rows": stage1_alert_count_all_rows,
    "stage2_alert_count_all_rows": stage2_alert_count_all_rows,
    "final_cascade_alert_count_all_rows": final_cascade_alert_count_all_rows,
    "stage1_alert_count_test_rows": stage1_alert_count_test_rows,
    "stage2_alert_count_test_rows": stage2_alert_count_test_rows,
    "final_cascade_alert_count_test_rows": final_cascade_alert_count_test_rows,
    "result_row_count": int(len(cascade_results)),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
}

cascade_metadata = {
    "cascade_results_path": str(CASCADE_RESULTS_PATH),
    "stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
    "stage1_models_path": str(STAGE1_MODELS_PATH),
    "stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
    "stage2_models_path": str(STAGE2_MODELS_PATH),
    "cascade_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
    "cascade_summary_path": str(CASCADE_SUMMARY_PATH),
    "cascade_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
    "gold_fit_path": str(GOLD_FIT_DATA_PATH),
    "gold_test_path": str(GOLD_TEST_DATA_PATH),
}

cascade_results.to_csv(CASCADE_RESULTS_PATH, index=False)
reference_profile.to_csv(CASCADE_REFERENCE_PROFILE_PATH, index=False)

joblib.dump(stage1_model, STAGE1_MODEL_ARTIFACT_PATH)
joblib.dump(stage1_model, STAGE1_MODELS_PATH)

joblib.dump(stage2_model, STAGE2_MODEL_ARTIFACT_PATH)
joblib.dump(stage2_model, STAGE2_MODELS_PATH)

save_json(cascade_thresholds, CASCADE_THRESHOLDS_PATH)
save_json(cascade_summary, CASCADE_SUMMARY_PATH)
save_json(cascade_metadata, CASCADE_METADATA_PATH)

wandb.save(str(CASCADE_RESULTS_PATH))
wandb.save(str(CASCADE_REFERENCE_PROFILE_PATH))
wandb.save(str(STAGE1_MODEL_ARTIFACT_PATH))
wandb.save(str(STAGE1_MODELS_PATH))
wandb.save(str(STAGE2_MODEL_ARTIFACT_PATH))
wandb.save(str(STAGE2_MODELS_PATH))
wandb.save(str(CASCADE_THRESHOLDS_PATH))
wandb.save(str(CASCADE_SUMMARY_PATH))
wandb.save(str(CASCADE_METADATA_PATH))

ledger.add(
    kind="step",
    step="save_cascade_outputs",
    message="Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, and reference profile.",
    data={
        "cascade_results_path": str(CASCADE_RESULTS_PATH),
        "cascade_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
        "stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "stage1_models_path": str(STAGE1_MODELS_PATH),
        "stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "stage2_models_path": str(STAGE2_MODELS_PATH),
        "cascade_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
        "cascade_summary_path": str(CASCADE_SUMMARY_PATH),
        "cascade_metadata_path": str(CASCADE_METADATA_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_test_path": str(GOLD_TEST_DATA_PATH),
        "result_row_count": int(len(cascade_results)),
        "stage1_alert_count_all_rows": stage1_alert_count_all_rows,
        "stage2_alert_count_all_rows": stage2_alert_count_all_rows,
        "final_cascade_alert_count_all_rows": final_cascade_alert_count_all_rows,
    },
    logger=logger,
)

2026-03-02 05:21:38,915 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_thresholds.json
2026-03-02 05:21:38,930 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_summary.json
2026-03-02 05:21:38,943 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_metadata.json
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
2026-03-02 05:21:39,181 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T05:21:39.181324+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'save_cascade_outputs', 'message': 'Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, and reference profile.', 'why': None, 'consequence': None, 'data': {'cascade_results_path': '/workspace/artifacts/gold/pump/pump

{'ts_utc': '2026-03-02T05:21:39.181324+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'save_cascade_outputs',
 'message': 'Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, and reference profile.',
 'why': None,
 'consequence': None,
 'data': {'cascade_results_path': '/workspace/artifacts/gold/pump/pump__gold__cascade_results.csv',
  'cascade_reference_profile_path': '/workspace/artifacts/gold/pump/pump__gold__cascade_reference_profile.csv',
  'stage1_model_path': '/workspace/artifacts/gold/pump/pump__gold__stage1_isolation_forest.joblib',
  'stage2_model_path': '/workspace/artifacts/gold/pump/pump__gold__stage2_isolation_forest.joblib',
  'cascade_thresholds_path': '/workspace/artifacts/gold/pump/pump__gold__cascade_thresholds.json',
  'cascade_summary_path': '/workspace/artifacts/gold/pump/pump__gold__cascade_summary.json',
  'cascade_metadata_path': '/workspace/artifacts/gold/pump/pump__gold__casc

In [ ]:
ledger.add(
    kind="step",
    step="finalize_cascade_modeling",
    message="Gold cascade modeling notebook complete.",
    data={
        "cascade_metrics": cascade_metrics,
        "cascade_results_path": str(CASCADE_RESULTS_PATH),
        "stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "stage1_models_path": str(STAGE1_MODELS_PATH),
        "stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "stage2_models_path": str(STAGE2_MODELS_PATH),
    },
    logger=logger,
)

cascade_ledger_path = GOLD_ARTIFACTS_PATH / GOLD_CASCADE_LEDGER_FILE_NAME
ledger.write_json(cascade_ledger_path)

wandb.save(str(cascade_ledger_path))
wandb_run.finish()

2026-03-02 05:21:39,204 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-02T05:21:39.204564+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'finalize_cascade_modeling', 'message': 'Gold cascade modeling notebook complete.', 'why': None, 'consequence': None, 'data': {'cascade_metrics': {'model': '3-Stage Cascade', 'stage1_alert_count_all_rows': 785, 'stage2_alert_count_all_rows': 2, 'final_alert_count_all_rows': 2, 'stage1_alert_count_test_rows': 785, 'stage2_alert_count_test_rows': 2, 'final_alert_count_test_rows': 2, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'cascade_results_path': '/workspace/artifacts/gold/pump/pump__gold__cascade_results.csv', 'stage1_model_path': '/workspace/artifacts/gold/pump/pump__gold__stage1_isolation_forest.joblib', 'stage2_model_path': '/workspace/artifacts/gold/pump/pump__gold__stage2_isolation_forest.joblib'}}
